# 🔍 PCB Component Detection - Interactive Testing Notebook
Upload any PCB photo (or select a test board) to instantly detect, count, and visualize all **18 component classes** using your trained model.

**Active Classes:** "IC", "LED", "battery", "capacitor", "connector", "diode", "display", "emi filter", "ferrite bead", "inductor", "mov", "potentiometer", "power capacitor", "relay", "resistor", "switch", "transistor", "transformer"


### Step 1: Install & Load Dependencies

In [ ]:
%pip install -q ultralytics matplotlib pillow

import os, glob, sys
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Locate the trained model weights
model_paths = [
    "models/pcb_best_model.pt",
    "best-2.pt",
    "best.pt",
    "pcb_best_model.pt"
]
model_file = next((p for p in model_paths if os.path.exists(p)), None)

# If running on Colab and model not uploaded yet, prompt upload
if not model_file and 'google.colab' in sys.modules:
    from google.colab import files
    print("📤 Model file not found. Please upload 'best-2.pt' or 'best.pt':")
    uploaded_model = files.upload()
    if uploaded_model:
        model_file = list(uploaded_model.keys())[0]

if model_file and os.path.exists(model_file):
    print(f"✅ Successfully loaded model: {model_file}")
    model = YOLO(model_file)
    print(f"🎯 Model has {len(model.names)} active classes: {list(model.names.values())}")
else:
    print("⚠️ Loading default backbone yolov8s.pt. For best results, place 'pcb_best_model.pt' in models/ or upload it.")
    model = YOLO("yolov8s.pt")


### Step 2: Upload Your PCB Image
Run this cell to upload a photo from your computer, or specify an existing image path.

In [ ]:
import sys

image_path = None

# Check if running inside Google Colab
if 'google.colab' in sys.modules:
    from google.colab import files
    print("📤 Click 'Choose Files' to upload your PCB image:")
    uploaded = files.upload()
    if uploaded:
        image_path = list(uploaded.keys())[0]
        print(f"✅ Successfully uploaded: {image_path}")
else:
    # Local execution: Pick a sample from test set or set your own path
    sample_images = glob.glob("dataset/test/images/*.jpg")
    default_sample = sample_images[0] if sample_images else "my_pcb.jpg"
    
    # CHANGE THIS to your image path if you have a custom photo:
    custom_path = ""  # e.g., "/Users/harmansingh/Desktop/my_board.jpg"
    image_path = custom_path if (custom_path and os.path.exists(custom_path)) else default_sample
    print(f"📸 Testing on image: {image_path}")

### Step 3: Run Detection & Component Breakdown

In [ ]:
if not image_path or not os.path.exists(image_path):
    print(f"❌ File '{image_path}' not found. Please upload or specify an image first.")
else:
    # Run prediction (Confidence threshold: 0.25)
    conf_threshold = 0.25
    results = model.predict(source=image_path, conf=conf_threshold, verbose=False)
    result = results[0]
    
    # Count detected classes
    counts = Counter()
    for box in result.boxes:
        cls_id = int(box.cls[0].item())
        cls_name = model.names[cls_id]
        counts[cls_name] += 1
        
    print("=" * 45)
    print(f"📊 DETECTION RESULTS: {os.path.basename(image_path)}")
    print(f"🎯 Total Components Detected: {sum(counts.values())}")
    print("=" * 45)
    for name, cnt in counts.most_common():
        print(f"  • {name:<18} : {cnt}")
    print("=" * 45)
    
    # Save annotated result image
    annotated_img_bgr = result.plot()
    annotated_img_rgb = annotated_img_bgr[..., ::-1]  # BGR to RGB
    output_filename = f"detected_{os.path.basename(image_path)}"
    Image.fromarray(annotated_img_rgb).save(output_filename)
    print(f"💾 Saved annotated image to: {output_filename}")

### Step 4: Display Side-by-Side Comparison

In [ ]:
if image_path and os.path.exists(image_path):
    orig_img = Image.open(image_path)
    det_img = Image.fromarray(annotated_img_rgb)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    axes[0].imshow(orig_img)
    axes[0].set_title(f"Original Image ({orig_img.size[0]}x{orig_img.size[1]})", fontsize=14)
    axes[0].axis("off")
    
    axes[1].imshow(det_img)
    axes[1].set_title(f"Detected Components (Total: {sum(counts.values())})", fontsize=14)
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()

### Step 5: Metallurgical Assay & Net Profit Valuation Range (₹ INR Only)
Calculates the recoverable precious and base metals as a **Min - Expected - Max range**, incorporates real-time commodity spot prices (Gold, Silver, Palladium, Copper) converted to Indian Rupees, subtracts stoichiometric reagent/extraction costs, and computes the **Net Profit Range in ₹ INR**.

In [ ]:
from metallurgy_engine import calculate_metallurgical_yield, get_component_range_profile

if "counts" in locals() and counts:
    valuation_report = calculate_metallurgical_yield(counts, use_live_prices=True)
    sm = valuation_report["summary"]
    gv_i = valuation_report["gross_valuation_inr"]
    ec_i = valuation_report["extraction_cost_inr"]
    np_i = valuation_report["net_profit_inr"]
    pm = valuation_report["net_profit_margin_percent"]

    print("=" * 70)
    print("⚗️  METALLURGICAL ASSAY & NET PROFIT RANGE VALUATION REPORT (₹ INR)")
    print("    Department of Metallurgical Engineering, IIT (BHU) Varanasi")
    print("=" * 70)
    print(f"📊 Market Feed Status : {sm['market_feed_status']} ({sm['market_feed_source']})")
    print(f"⏰ Pricing Timestamp  : {sm['market_feed_timestamp']}")
    print(f"⚖️ Est. Board Mass    : {sm['component_mass_g_range']['min']}g (Min) | {sm['component_mass_g_range']['avg']}g (Avg) | {sm['component_mass_g_range']['max']}g (Max)")
    print("-" * 70)
    print(f"{'Metal':<14} {'Recoverable Mass Range':<28} {'Gross Revenue (₹ INR)':<24}")
    print("-" * 70)
    for m, d in valuation_report["metals"].items():
        m_str = f"{d['recoverable_mass_mg_range']['min']} - {d['recoverable_mass_mg_range']['avg']} - {d['recoverable_mass_mg_range']['max']} mg"
        r_str = f"₹{d['gross_revenue_inr_range']['min']:,.2f} - ₹{d['gross_revenue_inr_range']['avg']:,.2f} - ₹{d['gross_revenue_inr_range']['max']:,.2f}"
        print(f"• {m:<12}: {m_str:<28} -> {r_str}")
    print("-" * 70)
    print("💰 FINANCIAL VALUATION PER PCB (RUPEES ONLY):")
    print(f"   • Gross Metal Value : ₹{gv_i['min']:,.2f} – ₹{gv_i['avg']:,.2f} – ₹{gv_i['max']:,.2f}")
    print(f"   • Extraction Cost   : ₹{ec_i['min']:,.2f} – ₹{ec_i['avg']:,.2f} – ₹{ec_i['max']:,.2f}  (Reagents, energy & labor)")
    print("   ------------------------------------------------------------------")
    print(f"   🚀 NET PROFIT RANGE : ₹{np_i['min']:,.2f} – ₹{np_i['avg']:,.2f} – ₹{np_i['max']:,.2f} INR / Board")
    print(f"   📈 Net Profit Margin: {pm['min']}% – {pm['avg']}% – {pm['max']}%")
    print("=" * 70)


### Step 6: Spotlight Analysis - Gold (Au) in Detected ICs & Leaching Reagents Required (₹ INR)
Specifically isolates the minimum, average, and maximum mass of Gold contained within the detected Integrated Circuits (ICs), along with stoichiometric leaching acid requirements.

In [ ]:
if "counts" in locals() and counts.get("IC", 0) > 0:
    ic_prof = get_component_range_profile("IC", "Au", count=counts["IC"])
    au = ic_prof["elements"]["Au"]
    print(f"🏆 SPOTLIGHT: GOLD (Au) RECOVERY FROM {counts['IC']} DETECTED ICs")
    print(f"   • Gold in 1 IC (Unit)   : {au['unit_mass_mg']['min']} mg (Min) | {au['unit_mass_mg']['avg']} mg (Avg) | {au['unit_mass_mg']['max']} mg (Max)")
    print(f"   • Total Gold in {counts['IC']} ICs: {au['total_mass_mg']['min']} mg (Min) | {au['total_mass_mg']['avg']} mg (Avg) | {au['total_mass_mg']['max']} mg (Max)")
    print(f"   • Recoverable Pure Gold : {au['recoverable_mass_mg']['min']} mg – {au['recoverable_mass_mg']['max']} mg (91% Extraction Yield)")
    print(f"   • Net Financial Gain    : ₹{au['recoverable_value_inr']['min']:,.2f} – ₹{au['recoverable_value_inr']['avg']:,.2f} – ₹{au['recoverable_value_inr']['max']:,.2f} INR")
    print("-" * 70)

rq = valuation_report["reagents_stoichiometry"]
print("🧪 HYDROMETALLURGICAL LEACHING REAGENTS REQUIRED:")
print(f"   • Conc. Nitric Acid (68% HNO3) : {rq['nitric_acid_68pct_ml_range']['min']} – {rq['nitric_acid_68pct_ml_range']['avg']} – {rq['nitric_acid_68pct_ml_range']['max']} mL")
print(f"   • Aqua Regia Solution (HCl+HNO3): {rq['aqua_regia_ml_range']['min']} – {rq['aqua_regia_ml_range']['avg']} – {rq['aqua_regia_ml_range']['max']} mL")
print(f"   • Sodium Metabisulfite (Na2S2O5): {rq['na2s2o5_precipitant_g_avg']:.3f} grams (Gold sponge reduction)")
